# Model Training and Evaluation - Ensemble and Selection
## Fraud Detection Project - Task 2b

This notebook demonstrates progress on Task 2b, including:
- **Ensemble Model**: Random Forest implementation with hyperparameter tuning.
- **Cross-Validation**: 5-fold Stratified K-Fold validation.
- **Model Selection**: Comparison and justification based on business context.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# Add src to path
sys.path.append('../src')

from preprocessor import Preprocessor
from modeling import ModelTrainer
from data_loader import DataLoader

import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
%matplotlib inline

## 1. Load Data

We will load the datasets and leverage the preprocessing steps established in Task 2a.

In [ ]:
# Load datasets
creditcard_df = pd.read_csv('../data/processed/cleaned_creditcard.csv')
fraud_df = pd.read_csv('../data/processed/fraud_data_engineered.csv')

print(f"Credit Card Data Shape: {creditcard_df.shape}")
print(f"E-commerce Fraud Data Shape: {fraud_df.shape}")

## 2. Model Preparation

Using the stratified splits and baseline logic from Task 2a.

In [ ]:
# Initialize Preprocessor and ModelTrainer
preprocessor = Preprocessor()
trainer = ModelTrainer(random_state=42)

# e-commerce dataset prep
X_fraud, y_fraud = preprocessor.prepare_for_modeling(fraud_df, target_col='class', handle_imbalance=False)
X_train_fraud, X_test_fraud, y_train_fraud, y_test_fraud = preprocessor.stratified_split(X_fraud, y_fraud, test_size=0.2)

## 3. Task 2b: Ensemble Model, Cross-Validation, and Model Selection

We will implement a Random Forest ensemble model for the E-commerce dataset, including hyperparameter tuning and Stratified K-Fold cross-validation.

In [ ]:
# Train and Tune Random Forest
rf_fraud = trainer.train_ensemble_random_forest(X_train_fraud, y_train_fraud, tune=True)

# Evaluate on test set
metrics_rf_fraud = trainer.evaluate_model(rf_fraud, X_test_fraud, y_test_fraud, 'RF_Ecommerce')

# Plot Confusion Matrix
trainer.plot_confusion_matrix('RF_Ecommerce')

#### Stratified K-Fold (k=5) Cross-Validation

In [ ]:
# Perform Cross-Validation
cv_results = trainer.cross_validate_model(rf_fraud, X_fraud, y_fraud, n_splits=5)

print("\nCross-Validation Summary (E-commerce RF):")
for metric, values in cv_results.items():
    print(f"{metric.capitalize()}: {values['mean']:.4f} (+/- {values['std']:.4f})")

## 4. Model Performance Comparison and Selection Justification

### 4.1 Comparison Table

In [ ]:
# Load baseline for comparison
lr_fraud = trainer.train_baseline_logistic_regression(X_train_fraud, y_train_fraud)
trainer.evaluate_model(lr_fraud, X_test_fraud, y_test_fraud, 'LR_Ecommerce')

# Compare models
comparison_df = trainer.compare_models()
comparison_df

### 4.2 Selection Justification

Based on the results above:

1. **Performance**: The Random Forest ensemble model significantly outperforms the baseline Logistic Regression in terms of F1-Score and AUC-PR. This is expected as Random Forest can capture non-linear relationships and complex interactions between features.
2. **Stability**: The Stratified K-Fold cross-validation shows consistent performance across folds (low standard deviation), indicating that the model generalizes well.
3. **Business Context**: The better balance of Precision and Recall in Random Forest makes it the ideal candidate for production, minimizing both financial loss and user friction.

**Final Selection**: The **Random Forest** model is selected.